# Interactive 3D landmark viewer (ATHENA)

Explore the 3D tracks produced by `athena_analysis.py`: choose **recording**, **landmarks**, **quantity** and
**time interval**, and see at once which data are **reliable**, **suspect**, **interpolated** or **missing**.

**How to use it**
1. Kernel **Python (athena)**:
   - in VSCode, *Select Kernel* → *Jupyter Kernel* → `Python (athena)`;
   - or from a terminal: `~/miniforge3/envs/athena/bin/jupyter lab notebooks/landmarks3d_viewer.ipynb`.
2. If needed, edit the **Parameters** cell, then *Run All*.
3. Use the controls in the last section. The functions can also be called from code: see *Use from code* at the end.

**Quality of each point** (file `analysis/<trial>/<trial>_3Dquality.npz`, written by `athena_analysis.py`):

| class | meaning |
|---|---|
| 🟩 reliable | triangulated (seen by ≥ 2 cameras) and passes all the checks below |
| 🟪 suspect | triangulated, but fails at least one check |
| 🟧 interpolated | seen by < 2 cameras: the value was filled in by the ATHENA smoothing (gaps of up to 5 frames) |
| 🟥 missing | no value in the final 3D (gap of 6 or more frames) |

**Checks** on triangulated points (thresholds in *Parameters* and in the *Quality thresholds* panel):

| check | fails when | what it reveals |
|---|---|---|
| `few_cams` | fewer than `MIN_CAMS` cameras see the point | no redundancy: with 2 cameras a wrong detection cannot be detected |
| `reproj` | max reprojection error over the cameras > `MAX_REPROJ_PX` | a camera whose 2D detection disagrees with the 3D point (wrong detection / wrong hand) |
| `outlier` | \|raw − final\| > `MAX_DEV_MM` | spike in the raw triangulation that the smoothing has flattened |
| `near_gap` | within half a smoothing window of a gap | the filter used ATHENA's linear fill across the gap: value partly distorted |

**Joint angles** (quantity *joint angles*, computed by `scripts/hand_angles.py`; degrees):

| angle | definition | sign |
|---|---|---|
| MCP / PIP / DIP flexion (index, middle, ring, pinky) | between proximal and distal segment (MCP: metacarpal → proximal phalanx), around the hand flexion axis (index MCP → pinky MCP) | 0 = straight, + flexion, − hyperextension |
| TAM | total active motion = MCP + PIP + DIP of a finger | |
| spread | between the proximal phalanges of two adjacent fingers, on the palm plane | ≥ 0 |
| thumb MCP / IP flexion | between consecutive thumb segments | 0 = straight, ≥ 0 |
| thumb abduction | between 1st (thumb) and 2nd (index) metacarpal | ≥ 0 |
| wrist flexion/extension | forearm (elbow → wrist) vs hand (wrist → middle MCP), around the flexion axis | + palmar flexion, − extension |
| wrist radial/ulnar deviation | same vectors on the palm plane | + radial, − ulnar |
| forearm rotation | palm normal vs vertical, around the forearm axis | 0 palm down … 180 palm up |

The quality of an angle in each frame is the **worst class among the landmarks it uses**. Wrist and forearm angles
use the elbow from MediaPipe Pose, which is less accurate than the hand landmarks.

## Parameters

In [1]:
# =========================== PARAMETERS (edit here) ===========================
from pathlib import Path

# project folder (contains scripts/athena_analysis.py): found automatically
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'athena_analysis.py').exists()), None)
assert PROJECT is not None, 'open the notebook from the DEXTER_kinematics project folder (or one of its subfolders)'

DATA_DIR   = PROJECT / 'data'        # folder with the recordings (e.g. Path.home() / 'remote_data' with SSHFS)
EXPORT_DIR = PROJECT / 'exports'     # where plots (HTML) and data (CSV) are saved

DEFAULT_RECORDING = 'finertapping'   # text contained in the recording name (None = the first one)
DEFAULT_GROUP     = 'right hand'     # 'right hand', 'left hand', 'body', 'all'
DEFAULT_LANDMARKS = ['R_thumb_tip', 'R_index_tip']
DEFAULT_QUANTITY  = 'position'       # one of QUANTITIES (next cell)
DEFAULT_AXES      = 'xyz'            # axes shown for position / velocity components
DEFAULT_ABC       = ('R_thumb_tip', 'R_index_tip', 'R_index_mcp')  # landmarks A, B, C for distance A–B and angle A–B–C
DEFAULT_ANGLES    = ['R index MCP flexion', 'R index PIP flexion']  # joint angles shown first (see ANGLE_NAMES)

# ---- reliability thresholds: a triangulated point failing any check is "suspect"
MIN_CAMS         = 3      # minimum number of cameras that see the point
MAX_REPROJ_PX    = 20.0   # max reprojection error over the cameras [px]
MAX_DEV_MM       = 10.0   # max |raw - final| [mm]; larger = spike flattened by the smoothing
EXCLUDE_NEAR_GAP = True   # frames within half a smoothing window of a gap are suspect

SMOOTH_FRAMES   = 1      # moving average on the positions before computing (1 = none)
SHOW_QUALITY    = True   # bands: orange = interpolated, red = missing
SHOW_SUSPECT    = True   # bands: purple = suspect
SHOW_RAW        = False  # overlay the raw triangulation (before the ATHENA smoothing): position and bone length
PLOT_HEIGHT     = 650    # plot height [px]
MAX_SHADED_RUNS = 400    # above this number of stretches the bands are not drawn (slow plot)
FPS_FALLBACK    = 40.0   # used only if the frame rate cannot be found

## Functions (loading, quality, computation, plots)

In [2]:
import json, re, sys, warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as W
from IPython.display import display, clear_output

sys.path.insert(0, str(PROJECT / 'scripts'))
import athena_analysis as AA   # landmark names, bones, raw triangulation, quality metrics
import hand_angles as HA       # joint angles (MCP, PIP, DIP, TAM, spread, thumb, wrist, forearm)

warnings.filterwarnings('ignore', category=RuntimeWarning)

LANDMARKS = list(AA.LANDMARK_NAMES)          # 0-32 body_*, 33-53 R_*, 54-74 L_*
IDX = {n: i for i, n in enumerate(LANDMARKS)}
GROUPS = {
    'right hand': [n for n in LANDMARKS if n.startswith('R_')],
    'left hand':  [n for n in LANDMARKS if n.startswith('L_')],
    'body':       [n for n in LANDMARKS if n.startswith('body_')],
    'all':        LANDMARKS,
}
CLASS = dict(AA.QUALITY_CLASS)               # 0 missing, 1 interpolated, 2 suspect, 3 reliable
CLASS_COLOR = {0: '#d62728', 1: '#ff7f0e', 2: '#9467bd', 3: '#2ca02c'}
REASONS = ['few_cams', 'reproj', 'outlier', 'near_gap']
QUANTITIES = ['position', 'speed (magnitude)', 'velocity (components)', 'acceleration (magnitude)',
              'joint angles', 'distance A–B', 'angle A–B–C', 'bone length', 'number of cameras',
              'reprojection error (max)', 'reprojection error (mean)', 'raw–final deviation']
UNITS = {'position': 'mm', 'speed (magnitude)': 'mm/s', 'velocity (components)': 'mm/s',
         'acceleration (magnitude)': 'mm/s²', 'joint angles': '°', 'distance A–B': 'mm', 'angle A–B–C': '°', 'bone length': 'mm',
         'number of cameras': 'cameras', 'reprojection error (max)': 'px', 'reprojection error (mean)': 'px',
         'raw–final deviation': 'mm'}
COLORS = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
SKELETON = ([(33 + a, 33 + b) for a, b in AA.HAND_BONES] + [(54 + a, 54 + b) for a, b in AA.HAND_BONES]
            + [(11, 12), (11, 13), (12, 14), (13, 54), (14, 33), (11, 23), (12, 24), (23, 24)])
PARENT = {LANDMARKS[o + b]: LANDMARKS[o + a] for o in (33, 54) for a, b in AA.HAND_BONES}  # bone child -> parent
ANGLE_NAMES = list(HA.all_hand_angles(np.zeros((1, 75, 3)))[0])   # 'R index MCP flexion', ..., 'L forearm rotation ...'
ANGLE_GROUPS = {'right hand': [a for a in ANGLE_NAMES if a.startswith('R ')],
                'left hand': [a for a in ANGLE_NAMES if a.startswith('L ')],
                'body': ANGLE_NAMES, 'all': ANGLE_NAMES}


# ------------------------------------------------------------------ loading
def find_trials(data_dir=None):
    """{name: (ATHENA main folder, trial)} for every 3D result available in data_dir."""
    data_dir = Path(data_dir or DATA_DIR).expanduser()
    files = []
    for pat in ('*/athena/landmarks/*/*_3Dlandmarks.npy', 'athena/landmarks/*/*_3Dlandmarks.npy',
                'landmarks/*/*_3Dlandmarks.npy'):
        files += sorted(data_dir.glob(pat))
    found = {}
    for f in files:
        main = f.parent.parent.parent
        rec = main.parent if main.name == 'athena' else main
        found.setdefault(rec.name, []).append((main, f.parent.name))
    return {(rec if len(items) == 1 else f'{rec} / {trial}'): (main, trial)
            for rec, items in found.items() for main, trial in items}


_CACHE = {}

def load_trial(name, data_dir=None):
    """Load final 3D and all per-point quality arrays; classify with the current thresholds.
    `name`: full name or part of the recording name."""
    trials = find_trials(data_dir)
    if name not in trials:
        hits = [k for k in trials if name.lower() in k.lower()]
        if len(hits) != 1:
            raise ValueError(f'"{name}": {len(hits)} matching recordings {hits[:5]}')
        name = hits[0]
    if name in _CACHE:
        return _CACHE[name]
    main, trial = trials[name]
    d3 = np.load(main / 'landmarks' / trial / f'{trial}_3Dlandmarks.npy')
    adir = main / 'analysis' / trial
    fps = None
    if (adir / 'run_info.json').exists():
        fps = float(json.loads((adir / 'run_info.json').read_text()).get('fps') or 0) or None
    if fps is None:
        try:
            fps = AA.video_frames(sorted((main / 'videos' / trial).glob('*.mp4'))[0])[1]
        except Exception:
            fps = FPS_FALLBACK
    qf = adir / f'{trial}_3Dquality.npz'
    q = dict(np.load(qf)) if qf.exists() else {}
    if 'reproj_max_px' not in q:  # quality file missing or from an older version
        print(f'{name}: computing the quality metrics (raw triangulation, reprojection) ...')
        q = AA.compute_quality(main, trial, d3=d3, fps=fps)
    tr = dict(name=name, main=main, trial=trial, d3=d3, fps=fps, t=np.arange(len(d3)) / fps,
              raw=np.asarray(q['raw3d'], float), status=q['status'], ncams=q['ncams'],
              reproj_max=np.asarray(q['reproj_max_px'], float), reproj_mean=np.asarray(q['reproj_mean_px'], float),
              dev=np.asarray(q['dev_mm'], float), near_gap=np.asarray(q['near_gap'], bool),
              window=int(q['smoothing_window']), q=q)
    classify(tr)
    _CACHE[name] = tr
    return tr


def classify(tr, min_cams=None, max_reproj_px=None, max_dev_mm=None, exclude_near_gap=None):
    """(Re)compute tr['cls'] (0 missing, 1 interpolated, 2 suspect, 3 reliable) and tr['reasons']."""
    thr = dict(min_cams=MIN_CAMS if min_cams is None else min_cams,
               max_reproj_px=MAX_REPROJ_PX if max_reproj_px is None else max_reproj_px,
               max_dev_mm=MAX_DEV_MM if max_dev_mm is None else max_dev_mm,
               exclude_near_gap=EXCLUDE_NEAR_GAP if exclude_near_gap is None else exclude_near_gap)
    if tr.get('thr') != thr:
        tr['cls'], tr['reasons'] = AA.quality_classes(tr['q'], **thr)
        tr['thr'] = thr
    return tr


def reason_text(tr, sl, i):
    """Per frame: names of the failed checks for landmark index i, e.g. 'few_cams;reproj'."""
    flags = np.stack([tr['reasons'][r][sl, i] for r in REASONS], -1)
    return np.array([';'.join(r for r, f in zip(REASONS, row) if f) for row in flags], dtype=object)


def worst_status(tr, sl, lms):
    """Per frame: worst class among the landmarks lms (codes) and the union of their failed checks (text)."""
    idx = [IDX[n] if isinstance(n, str) else n for n in lms]
    codes = tr['cls'][sl][:, idx].min(1)
    flags = np.zeros((len(codes), len(REASONS)), bool)
    for k, r in enumerate(REASONS):
        flags[:, k] = tr['reasons'][r][sl][:, idx].any(1)
    text = np.array([';'.join(r for r, f in zip(REASONS, row) if f) for row in flags], dtype=object)
    return codes, text


def trial_angles(tr, smooth=1):
    """Joint angles of both hands (cached per moving-average width): ({name: (frames,)}, {name: [landmark idx]})."""
    key = ('angles', smooth)
    if key not in tr:
        tr[key] = HA.all_hand_angles(moving_avg(tr['d3'], smooth))
    return tr[key]


# ------------------------------------------------------------------ computation
def frame_slice(tr, t_range=None):
    n = len(tr['t'])
    if t_range is None:
        return slice(0, n)
    a = max(0, int(np.floor(t_range[0] * tr['fps'] + 1e-6)))
    b = min(n, int(np.floor(t_range[1] * tr['fps'] + 1e-6)) + 1)
    return slice(a, max(b, a + 1))


def moving_avg(y, n):
    """Centered moving average over n frames (NaNs stay NaN)."""
    if n <= 1:
        return y
    sm = pd.DataFrame(y.reshape(len(y), -1)).rolling(n, center=True, min_periods=1).mean().to_numpy()
    sm = np.array(sm, dtype=float).reshape(y.shape)  # writable copy
    sm[np.isnan(y)] = np.nan
    return sm


def runs(mask):
    """[(start, end_exclusive)] of the True stretches."""
    d = np.diff(np.concatenate([[0], np.asarray(mask, int), [0]]))
    return list(zip(np.where(d == 1)[0], np.where(d == -1)[0]))


def compute_series(tr, landmarks, quantity='position', axes='xyz', A=None, B=None, C=None, t_range=None, smooth=1,
                   angles=None):
    """Compute the requested quantity in the interval.
    Returns (DataFrame[frame, time_s, <series>..., status_<x>, flags_<x>...], series columns, landmarks involved).
    status = reliable / suspect / interpolated / missing; flags = failed checks of suspect points. For quantities
    derived from several landmarks (joint angles, distance, angle, bone length) status/flags are given per series
    (worst class among its landmarks); df.attrs['col_landmarks'] = {series: landmarks used}."""
    sl = frame_slice(tr, t_range)
    d3, fps = tr['d3'], tr['fps']
    df = pd.DataFrame({'frame': np.arange(len(d3))[sl], 'time_s': tr['t'][sl]})
    pos = lambda n: moving_avg(d3[:, IDX[n]], smooth)
    ax = {'x': 0, 'y': 1, 'z': 2}
    cols, col_lms = [], {}
    if quantity == 'joint angles':
        Aang, Lang = trial_angles(tr, smooth)
        involved = []
        for a in (angles or []):
            cols.append(a); df[a] = Aang[a][sl]
            col_lms[a] = [LANDMARKS[i] for i in Lang[a]]
            involved += [n for n in col_lms[a] if n not in involved]
    elif quantity == 'distance A–B':
        involved = [A, B]
        cols = [f'dist {A}–{B}']
        df[cols[0]] = np.linalg.norm(pos(A) - pos(B), axis=1)[sl]
    elif quantity == 'angle A–B–C':
        involved = [A, B, C]
        u, w = pos(A) - pos(B), pos(C) - pos(B)
        cos = (u * w).sum(1) / (np.linalg.norm(u, axis=1) * np.linalg.norm(w, axis=1))
        cols = [f'angle {A}–{B}–{C}']
        df[cols[0]] = np.degrees(np.arccos(np.clip(cos, -1, 1)))[sl]
    elif quantity == 'bone length':
        involved = []
        for n in landmarks:
            if n in PARENT:  # bone parent -> n (wrist and body landmarks have no bone)
                involved += [x for x in (PARENT[n], n) if x not in involved]
                cols.append(f'{n}_bone'); df[cols[-1]] = np.linalg.norm(pos(n) - pos(PARENT[n]), axis=1)[sl]
                col_lms[cols[-1]] = [PARENT[n], n]
    else:
        involved = list(landmarks)
        for n in involved:
            i = IDX[n]
            if quantity == 'position':
                p = pos(n)
                for a in axes:
                    cols.append(f'{n}_{a}'); df[cols[-1]] = p[sl, ax[a]]
            elif quantity == 'number of cameras':
                cols.append(f'{n}_ncams'); df[cols[-1]] = tr['ncams'][sl, i]
            elif quantity == 'reprojection error (max)':
                cols.append(f'{n}_reproj_max'); df[cols[-1]] = tr['reproj_max'][sl, i]
            elif quantity == 'reprojection error (mean)':
                cols.append(f'{n}_reproj_mean'); df[cols[-1]] = tr['reproj_mean'][sl, i]
            elif quantity == 'raw–final deviation':
                cols.append(f'{n}_dev'); df[cols[-1]] = tr['dev'][sl, i]
            else:
                v = np.gradient(pos(n), axis=0) * fps
                if quantity == 'acceleration (magnitude)':
                    cols.append(f'{n}_|a|'); df[cols[-1]] = np.linalg.norm(np.gradient(moving_avg(v, smooth), axis=0) * fps, axis=1)[sl]
                elif quantity == 'velocity (components)':
                    for a in axes:
                        cols.append(f'{n}_v{a}'); df[cols[-1]] = v[sl, ax[a]]
                else:
                    cols.append(f'{n}_|v|'); df[cols[-1]] = np.linalg.norm(v, axis=1)[sl]
    if quantity in ('distance A–B', 'angle A–B–C'):
        col_lms[cols[0]] = list(involved)
    for c in cols:
        if c not in col_lms:   # per-landmark series: '<landmark>_<suffix>'
            col_lms[c] = [next(n for n in involved if c.startswith(n + '_'))]
    if quantity in ('joint angles', 'distance A–B', 'angle A–B–C', 'bone length'):
        for c in cols:   # one status per series (worst among its landmarks)
            codes, text = worst_status(tr, sl, col_lms[c])
            df[f'status_{c}'] = pd.Series(codes).map(CLASS).to_numpy()
            df[f'flags_{c}'] = text
    if quantity != 'joint angles':
        for n in involved:
            df[f'status_{n}'] = pd.Series(tr['cls'][sl, IDX[n]]).map(CLASS).to_numpy()
            df[f'flags_{n}'] = reason_text(tr, sl, IDX[n])
    df.attrs['col_landmarks'] = col_lms
    return df, cols, involved


def quality_counts(tr, landmarks, t_range=None):
    c = tr['cls'][frame_slice(tr, t_range)][:, [IDX[n] for n in landmarks]]
    return {CLASS[k]: 100 * float((c == k).mean()) for k in (3, 2, 1, 0)}


# ------------------------------------------------------------------ plots
def _empty(msg):
    fig = go.Figure()
    fig.add_annotation(text=msg, showarrow=False, font=dict(size=16))
    fig.update_layout(height=250, xaxis_visible=False, yaxis_visible=False)
    return fig


def _shade_quality(fig, tr, sl, involved, show_suspect=True):
    """Bands of the worst class among the involved landmarks: purple suspect, orange interpolated, red missing."""
    worst = tr['cls'][sl][:, [IDX[n] for n in involved]].min(1)
    t, h = tr['t'][sl], 0.5 / tr['fps']
    for code in ((2, 1, 0) if show_suspect else (1, 0)):
        rr = runs(worst == code)
        if not rr:
            continue
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=f'{CLASS[code]} ({len(rr)} stretches)',
                                 marker=dict(symbol='square', size=12, color=CLASS_COLOR[code], opacity=0.4)))
        if len(rr) > MAX_SHADED_RUNS:
            continue
        for a, b in rr:
            fig.add_vrect(x0=t[a] - h, x1=t[b - 1] + h, fillcolor=CLASS_COLOR[code], opacity=0.15 if code == 2 else 0.18,
                          line_width=0, layer='below', row='all', col=1)


def _threshold_line(fig, quantity, nrows):
    thr = {'reprojection error (max)': MAX_REPROJ_PX, 'raw–final deviation': MAX_DEV_MM,
           'number of cameras': MIN_CAMS}.get(quantity)
    if thr is not None:
        fig.add_hline(y=thr, line_dash='dash', line_color='gray', annotation_text='threshold',
                      annotation_position='top left', row=nrows, col=1)


def timeseries_figure(tr, landmarks, quantity='position', axes='xyz', A=None, B=None, C=None, t_range=None,
                      smooth=1, show_quality=True, show_suspect=True, show_raw=False, height=None, angles=None):
    """Time series of the chosen quantity; position and velocity components: one panel per axis."""
    A, B, C = A or DEFAULT_ABC[0], B or DEFAULT_ABC[1], C or DEFAULT_ABC[2]
    axes = axes or 'xyz'
    df, cols, involved = compute_series(tr, landmarks, quantity, axes, A, B, C, t_range, smooth, angles)
    if not cols:
        return _empty('Select at least one joint angle' if quantity == 'joint angles' else
                      'Select at least one landmark' + (' with a bone (not the wrist)' if quantity == 'bone length' else ''))
    sl = frame_slice(tr, t_range)
    by_axis = quantity in ('position', 'velocity (components)')
    nrows = len(axes) if by_axis else 1
    unit = UNITS[quantity]
    fig = make_subplots(rows=nrows, cols=1, shared_xaxes=True, vertical_spacing=0.05,
                        subplot_titles=[f'{a}  [{unit}]' for a in axes] if by_axis and nrows > 1 else None)
    seen = set()
    col_lms = df.attrs['col_landmarks']
    for k, col in enumerate(cols):
        lm = col_lms[col][-1] if quantity not in ('joint angles', 'distance A–B', 'angle A–B–C') else None
        key = lm or col
        color = COLORS[(landmarks.index(lm) if lm in landmarks else k) % len(COLORS)]
        row = axes.index(col[-1]) + 1 if by_axis else 1
        sk = col if f'status_{col}' in df else lm   # per-series status for derived quantities, else per landmark
        st, fl = df[f'status_{sk}'], df[f'flags_{sk}']
        fig.add_trace(go.Scattergl(
            x=df.time_s, y=df[col], mode='lines', name=key, legendgroup=key, showlegend=key not in seen,
            line=dict(color=color, width=1.6), customdata=np.stack([df.frame, st, fl], -1),
            hovertemplate=f'<b>{col}</b><br>t = %{{x:.3f}} s (frame %{{customdata[0]}})<br>'
                          f'%{{y:.2f}} {unit}<br>%{{customdata[1]}} %{{customdata[2]}}<extra></extra>'), row=row, col=1)
        seen.add(key)
        if show_raw and lm and quantity in ('position', 'bone length'):
            raw = tr['raw']
            y = (raw[sl, IDX[lm], 'xyz'.index(col[-1])] if quantity == 'position'
                 else np.linalg.norm(raw[sl, IDX[lm]] - raw[sl, IDX[PARENT[lm]]], axis=1))
            fig.add_trace(go.Scattergl(
                x=df.time_s, y=y, mode='markers', name=f'{lm} raw', legendgroup=key, showlegend=False,
                marker=dict(color=color, size=3, opacity=0.5),
                hovertemplate=f'{col} raw: %{{y:.2f}} {unit}<extra></extra>'), row=row, col=1)
    if show_quality:
        _shade_quality(fig, tr, sl, involved, show_suspect)
    _threshold_line(fig, quantity, nrows)
    q = quality_counts(tr, involved, t_range)
    fig.update_layout(
        height=height or PLOT_HEIGHT, hovermode='closest', margin=dict(t=70, l=60, r=20, b=40),
        title=dict(text=f'<b>{tr["name"]}</b> – {quantity}<br><sup>{df.time_s.iloc[0]:.2f}–{df.time_s.iloc[-1]:.2f} s, '
                        f'{len(df)} frames  |  ' + ', '.join(f'{k} {v:.1f}%' for k, v in q.items()) + '</sup>'))
    fig.update_xaxes(title_text='time [s]', row=nrows, col=1)
    if not by_axis or nrows == 1:
        fig.update_yaxes(title_text=f'{quantity} [{unit}]' + (f' – {axes}' if by_axis else ''))
    return fig


def trajectory_figure(tr, landmarks, t_range=None, frame=None, skeleton=True, height=None):
    """3D trajectories in the interval (colour = time) + hand skeleton at the chosen frame, joints coloured by class."""
    sl = frame_slice(tr, t_range)
    t = tr['t'][sl]
    fig = go.Figure()
    for i, n in enumerate(landmarks):
        p = tr['d3'][sl, IDX[n]]
        fig.add_trace(go.Scatter3d(
            x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='lines+markers', name=n,
            line=dict(color=COLORS[i % len(COLORS)], width=3),
            marker=dict(size=2.5, color=t, colorscale='Viridis', showscale=(i == 0),
                        colorbar=dict(title='t [s]', len=0.6) if i == 0 else None),
            customdata=t, hovertemplate=f'{n}<br>t=%{{customdata:.3f}} s<br>x=%{{x:.1f}} y=%{{y:.1f}} z=%{{z:.1f}}<extra></extra>'))
    if skeleton and frame is not None:
        P = tr['d3'][frame]
        xs, ys, zs = [], [], []
        for a, b in SKELETON:
            xs += [P[a, 0], P[b, 0], None]; ys += [P[a, 1], P[b, 1], None]; zs += [P[a, 2], P[b, 2], None]
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode='lines', name=f'skeleton frame {frame}',
                                   line=dict(color='gray', width=5), hoverinfo='skip'))
        hands = list(range(33, 75))
        c = tr['cls'][frame, hands]
        text = []
        for h, k in zip(hands, c):
            fl = ';'.join(r for r in REASONS if tr['reasons'][r][frame, h])
            text.append(f'{LANDMARKS[h]}: {CLASS[k]} {fl}<br>{tr["ncams"][frame, h]} cameras, '
                        f'reproj max {tr["reproj_max"][frame, h]:.1f} px, dev {tr["dev"][frame, h]:.1f} mm')
        fig.add_trace(go.Scatter3d(x=P[hands, 0], y=P[hands, 1], z=P[hands, 2], mode='markers', name='joints (class)',
                                   marker=dict(size=4, color=[CLASS_COLOR[k] for k in c]), text=text, hoverinfo='text'))
    if not fig.data:
        return _empty('Select at least one landmark')
    fig.update_layout(height=height or PLOT_HEIGHT, margin=dict(t=50, l=0, r=0, b=0),
                      title=f'<b>{tr["name"]}</b> – 3D trajectories {t[0]:.2f}–{t[-1]:.2f} s'
                            + (f', skeleton at frame {frame} ({frame / tr["fps"]:.2f} s)' if frame is not None else ''),
                      scene=dict(aspectmode='data', xaxis_title='X [mm]', yaxis_title='Y [mm]', zaxis_title='Z [mm]'))
    return fig


HEATMAPS = ['class', 'cameras', 'reprojection error (max)', 'raw–final deviation']

def quality_heatmap(tr, landmarks, t_range=None, what='class', height=None):
    """Map over time, per landmark, of: class, number of cameras, max reprojection error or raw–final deviation."""
    if not landmarks:
        return _empty('Select at least one landmark')
    sl = frame_slice(tr, t_range)
    idx = [IDX[n] for n in landmarks]
    t = tr['t'][sl]
    if what == 'class':
        z = tr['cls'][sl][:, idx].T
        cs = []
        for k in range(4):
            cs += [[k / 4, CLASS_COLOR[k]], [(k + 1) / 4, CLASS_COLOR[k]]]
        reasons = np.stack([reason_text(tr, sl, i) for i in idx])
        hm = go.Heatmap(z=z, x=t, y=landmarks, zmin=-0.5, zmax=3.5, colorscale=cs, customdata=reasons,
                        colorbar=dict(tickvals=[0, 1, 2, 3], ticktext=[CLASS[k] for k in range(4)]),
                        hovertemplate='%{y}<br>t=%{x:.3f} s<br>%{customdata}<extra></extra>')
        title = 'point class'
    elif what == 'cameras':
        z = tr['ncams'][sl][:, idx].T
        hm = go.Heatmap(z=z, x=t, y=landmarks, colorscale='Viridis', zmin=0, zmax=int(tr['ncams'].max()),
                        colorbar=dict(title='cameras'), hovertemplate='%{y}<br>t=%{x:.3f} s<br>%{z} cameras<extra></extra>')
        title = 'cameras seeing the point'
    else:
        key, thr, unit = (('reproj_max', MAX_REPROJ_PX, 'px') if what.startswith('reprojection')
                          else ('dev', MAX_DEV_MM, 'mm'))
        z = tr[key][sl][:, idx].T
        hm = go.Heatmap(z=z, x=t, y=landmarks, colorscale='Inferno', zmin=0, zmax=3 * thr,
                        colorbar=dict(title=f'{unit}<br>(max = 3× threshold)'),
                        hovertemplate=f'%{{y}}<br>t=%{{x:.3f}} s<br>%{{z:.1f}} {unit}<extra></extra>')
        title = f'{what} (threshold {thr:g} {unit})'
    fig = go.Figure(hm)
    fig.update_layout(height=height or max(300, 22 * len(landmarks) + 120), margin=dict(t=50, l=160, r=20, b=40),
                      title=f'<b>{tr["name"]}</b> – {title}', xaxis_title='time [s]', yaxis=dict(autorange='reversed'))
    return fig


def stats_table(tr, landmarks, t_range=None):
    """Per-landmark statistics in the interval: classes, failed checks, cameras, errors, gaps, movement, speed."""
    sl = frame_slice(tr, t_range)
    fps, rows = tr['fps'], []
    for n in landmarks:
        i = IDX[n]
        c, nc, p = tr['cls'][sl, i], tr['ncams'][sl, i], tr['d3'][sl, i]
        tri = c >= 2
        v = np.linalg.norm(np.gradient(tr['d3'][:, i], axis=0) * fps, axis=1)[sl]
        gaps = runs(c < 2)
        row = {'landmark': n,
               '% reliable': 100 * (c == 3).mean(), '% suspect': 100 * (c == 2).mean(),
               '% interpolated': 100 * (c == 1).mean(), '% missing': 100 * (c == 0).mean()}
        row.update({f'% {r}': 100 * tr['reasons'][r][sl, i].mean() for r in REASONS})
        row.update({
            'mean cameras': nc[tri].mean() if tri.any() else np.nan,
            'median reproj max [px]': np.nanmedian(tr['reproj_max'][sl, i]),
            'p99 raw–final dev [mm]': np.nanpercentile(tr['dev'][sl, i], 99) if np.isfinite(tr['dev'][sl, i]).any() else np.nan,
            'n gaps': len(gaps), 'max gap [s]': max((b - a for a, b in gaps), default=0) / fps,
            'range x [mm]': np.nanmax(p[:, 0]) - np.nanmin(p[:, 0]) if np.isfinite(p[:, 0]).any() else np.nan,
            'range y [mm]': np.nanmax(p[:, 1]) - np.nanmin(p[:, 1]) if np.isfinite(p[:, 1]).any() else np.nan,
            'range z [mm]': np.nanmax(p[:, 2]) - np.nanmin(p[:, 2]) if np.isfinite(p[:, 2]).any() else np.nan,
            'mean speed [mm/s]': np.nanmean(v), 'max speed [mm/s]': np.nanmax(v) if np.isfinite(v).any() else np.nan,
        })
        rows.append(row)
    return pd.DataFrame(rows).set_index('landmark').round(2) if rows else pd.DataFrame()


def angle_stats(tr, angles, t_range=None, smooth=1):
    """Per joint angle in the interval: quality (worst class among its landmarks) and range of motion,
    on all frames and on the reliable frames only."""
    sl = frame_slice(tr, t_range)
    Aang, Lang = trial_angles(tr, smooth)
    pct = lambda x, q: np.nanpercentile(x, q) if len(x) and np.isfinite(x).any() else np.nan
    rows = []
    for a in angles:
        v = Aang[a][sl]
        codes, _ = worst_status(tr, sl, Lang[a])
        ok = np.isfinite(v)
        vr = v[(codes == 3) & ok]
        rows.append({'angle': a,
                     '% reliable': 100 * (codes == 3).mean(), '% suspect': 100 * (codes == 2).mean(),
                     '% interpolated': 100 * (codes == 1).mean(), '% missing': 100 * (codes == 0).mean(),
                     'mean [°]': np.nanmean(v) if ok.any() else np.nan, 'SD [°]': np.nanstd(v) if ok.any() else np.nan,
                     'min [°]': np.nanmin(v) if ok.any() else np.nan, 'max [°]': np.nanmax(v) if ok.any() else np.nan,
                     'ROM [°]': (np.nanmax(v) - np.nanmin(v)) if ok.any() else np.nan,
                     'ROM p5–p95 [°]': pct(v, 95) - pct(v, 5),
                     'mean, reliable [°]': np.mean(vr) if len(vr) else np.nan,
                     'ROM p5–p95, reliable [°]': pct(vr, 95) - pct(vr, 5)})
    return pd.DataFrame(rows).set_index('angle').round(1) if rows else pd.DataFrame()


def safe_name(s):
    return re.sub(r'[^\w.-]+', '_', s).strip('_')


print(f'{len(find_trials())} recordings with 3D data in {DATA_DIR}')

7 recordings with 3D data in /home/roberto/Desktop/DEXTER_kinematics/data


## Interactive viewer

- **Group** filters the landmark list. Use Ctrl/Shift + click to select more than one.
- **Quantity**:
  - position, speed (magnitude) or velocity (components), acceleration;
  - **joint angles** (MCP, PIP, DIP flexion, TAM, finger spread, thumb, wrist, forearm rotation): chosen in the
    *Joint angles* list, which replaces the landmark list for this quantity;
  - distance between two landmarks (A–B) and angle at B (A–B–C) between any three landmarks;
  - **bone length** ending at each selected landmark (e.g. `R_index_tip` = distal phalanx): it should be constant;
  - quality: number of cameras, **reprojection error** (max / mean over the cameras), **raw–final deviation**.
    The dashed line is the threshold.
- **Quality thresholds** panel: change the reliability checks; all plots and tables update.
- **Interval**: drag the ends or press *Whole track*. You can also zoom with the mouse inside the plots.
- **Tabs**:
  - *Time series*: purple bands are suspect, orange interpolated, red missing; hover shows the failed checks;
  - *3D trajectory*: skeleton at the chosen frame, joints coloured by class;
  - *Quality*: map over time of class, cameras, reprojection error or raw–final deviation;
  - *Statistics*: table per landmark, including the share of each failed check; with *joint angles*, table per angle
    with quality and range of motion (all frames and reliable frames only).
- **Save** writes the plot (interactive HTML) or the data (CSV, with class and failed checks of every point) to `EXPORT_DIR`.

In [3]:
TRIALS = find_trials()
assert TRIALS, f'no 3D data found in {DATA_DIR}: process the recordings with athena_analysis.py first'
names = list(TRIALS)
rec0 = next((n for n in names if DEFAULT_RECORDING and DEFAULT_RECORDING.lower() in n.lower()), names[0])
lay = lambda w: W.Layout(width=w)
sty = dict(description_width='110px')

w_rec = W.Dropdown(options=names, value=rec0, description='Recording', layout=lay('560px'), style=sty)
w_group = W.ToggleButtons(options=list(GROUPS), value=DEFAULT_GROUP, description='Group', style=dict(description_width='110px', button_width='110px'))
w_lms = W.SelectMultiple(options=GROUPS[DEFAULT_GROUP], value=[n for n in DEFAULT_LANDMARKS if n in GROUPS[DEFAULT_GROUP]],
                         rows=16, description='Landmarks', layout=lay('330px'), style=sty)
w_angles = W.SelectMultiple(options=ANGLE_GROUPS[DEFAULT_GROUP], value=[a for a in DEFAULT_ANGLES if a in ANGLE_GROUPS[DEFAULT_GROUP]],
                            rows=16, description='Joint angles', layout=lay('380px'), style=sty)
w_qty = W.Dropdown(options=QUANTITIES, value=DEFAULT_QUANTITY, description='Quantity', layout=lay('380px'), style=sty)
w_axes = [W.Checkbox(value=a in DEFAULT_AXES, description=a, indent=False, layout=lay('45px')) for a in 'xyz']
w_axes_box = W.HBox([W.Label('Axes', layout=lay('110px'))] + w_axes)
w_A = W.Dropdown(options=LANDMARKS, value=DEFAULT_ABC[0], description='A', layout=lay('380px'), style=sty)
w_B = W.Dropdown(options=LANDMARKS, value=DEFAULT_ABC[1], description='B', layout=lay('380px'), style=sty)
w_C = W.Dropdown(options=LANDMARKS, value=DEFAULT_ABC[2], description='C (angle at B)', layout=lay('380px'), style=sty)
w_smooth = W.IntSlider(value=SMOOTH_FRAMES, min=1, max=21, step=2, description='Moving average', continuous_update=False,
                       layout=lay('380px'), style=sty)
w_quality = W.Checkbox(value=SHOW_QUALITY, description='Highlight interpolated / missing', indent=False)
w_suspect = W.Checkbox(value=SHOW_SUSPECT, description='Highlight suspect', indent=False)
w_raw = W.Checkbox(value=SHOW_RAW, description='Show raw triangulation', indent=False)
# reliability thresholds
w_mincams = W.IntSlider(value=MIN_CAMS, min=2, max=9, description='Min cameras', continuous_update=False, style=sty, layout=lay('380px'))
w_reproj = W.FloatSlider(value=MAX_REPROJ_PX, min=2, max=100, step=1, description='Max reproj [px]', continuous_update=False, style=sty, layout=lay('380px'))
w_dev = W.FloatSlider(value=MAX_DEV_MM, min=1, max=50, step=0.5, description='Max dev [mm]', continuous_update=False, style=sty, layout=lay('380px'))
w_neargap = W.Checkbox(value=EXCLUDE_NEAR_GAP, description='Frames near a gap are suspect', indent=False)
w_thr = W.Accordion(children=[W.VBox([w_mincams, w_reproj, w_dev, w_neargap])], layout=lay('400px'))
w_thr.set_title(0, 'Quality thresholds')
w_time = W.FloatRangeSlider(min=0, max=1, step=0.025, value=(0, 1), description='Interval [s]', continuous_update=False,
                            readout_format='.2f', layout=lay('900px'), style=sty)
w_full = W.Button(description='Whole track', icon='arrows-h')
w_frame = W.IntSlider(description='Skeleton frame', continuous_update=False, layout=lay('900px'), style=sty)
w_heat = W.ToggleButtons(options=HEATMAPS, description='Map of', style=dict(description_width='110px', button_width='170px'))
w_heat_group = W.Checkbox(value=True, description='Whole group (not only the selected landmarks)', indent=False)
w_save_html = W.Button(description='Save plot (HTML)', icon='save')
w_save_csv = W.Button(description='Save data (CSV)', icon='table')
w_info, w_msg = W.HTML(), W.HTML()
out_ts, out_3d, out_heat, out_stats = W.Output(), W.Output(), W.Output(), W.Output()
tabs = W.Tab(children=[out_ts, W.VBox([w_frame, out_3d]), W.VBox([w_heat, w_heat_group, out_heat]), out_stats])
tabs.titles = ('Time series', '3D trajectory', 'Quality', 'Statistics')
STATE = {'busy': False, 'fig': None}


def current_trial():
    """Trial of the dropdown, classified with the thresholds of the panel."""
    return classify(load_trial(w_rec.value), w_mincams.value, w_reproj.value, w_dev.value, w_neargap.value)


def params():
    return dict(tr=current_trial(), landmarks=list(w_lms.value), quantity=w_qty.value,
                axes=''.join(c.description for c in w_axes if c.value) or 'xyz', A=w_A.value, B=w_B.value, C=w_C.value,
                t_range=w_time.value, smooth=w_smooth.value, angles=list(w_angles.value))


def update_visibility():
    q = w_qty.value
    w_angles.layout.display = '' if q == 'joint angles' else 'none'
    w_lms.layout.display = 'none' if q == 'joint angles' else ''
    w_axes_box.layout.display = '' if q in ('position', 'velocity (components)') else 'none'
    w_A.layout.display = w_B.layout.display = '' if q in ('distance A–B', 'angle A–B–C') else 'none'
    w_C.layout.display = '' if q == 'angle A–B–C' else 'none'
    w_raw.layout.display = '' if q in ('position', 'bone length') else 'none'


def redraw(*_):
    if STATE['busy']:
        return
    update_visibility()
    global MIN_CAMS, MAX_REPROJ_PX, MAX_DEV_MM, EXCLUDE_NEAR_GAP   # thresholds used by plots (threshold lines)
    MIN_CAMS, MAX_REPROJ_PX, MAX_DEV_MM, EXCLUDE_NEAR_GAP = w_mincams.value, w_reproj.value, w_dev.value, w_neargap.value
    p = params()
    tr = p['tr']
    sl = frame_slice(tr, p['t_range'])
    w_info.value = (f"<b>{tr['name']}</b> &nbsp; {len(tr['t'])} frames @ {tr['fps']:g} fps = {len(tr['t']) / tr['fps']:.2f} s"
                    f" &nbsp;|&nbsp; interval: frames {sl.start}–{sl.stop - 1} ({sl.stop - sl.start} frames)"
                    f" &nbsp;|&nbsp; smoothing window {tr['window']} frames")
    i = tabs.selected_index or 0
    try:
        if i == 0:
            fig = timeseries_figure(**p, show_quality=w_quality.value, show_suspect=w_suspect.value, show_raw=w_raw.value)
            target = out_ts
        elif i == 1:
            fig = trajectory_figure(p['tr'], p['landmarks'], p['t_range'], frame=w_frame.value)
            target = out_3d
        elif i == 2:
            lms = GROUPS[w_group.value] if w_heat_group.value else p['landmarks']
            fig = quality_heatmap(p['tr'], lms, p['t_range'], what=w_heat.value)
            target = out_heat
        else:
            fig = None
            with out_stats:
                clear_output(wait=True)
                if p['quantity'] == 'joint angles':
                    display(angle_stats(p['tr'], p['angles'] or ANGLE_GROUPS[w_group.value], p['t_range'], p['smooth']))
                else:
                    display(stats_table(p['tr'], p['landmarks'] or GROUPS[w_group.value], p['t_range']))
        if fig is not None:
            STATE['fig'] = fig
            with target:
                clear_output(wait=True)
                fig.show()
    except Exception as e:
        w_msg.value = f'<span style="color:#d62728">error: {type(e).__name__}: {e}</span>'
    else:
        if w_msg.value.startswith('<span style="color:#d62728">'):
            w_msg.value = ''


def on_recording(*_):
    tr = load_trial(w_rec.value)
    dur = (len(tr['t']) - 1) / tr['fps']
    STATE['busy'] = True
    w_time.step = 1 / tr['fps']
    w_time.max = dur
    w_time.min = 0
    w_time.value = (0, dur)
    STATE['busy'] = False
    on_time()


def on_time(*_):
    tr = load_trial(w_rec.value)
    sl = frame_slice(tr, w_time.value)
    STATE['busy'] = True
    w_frame.min, w_frame.max = 0, len(tr['t']) - 1
    w_frame.min, w_frame.max = sl.start, sl.stop - 1
    w_frame.value = (sl.start + sl.stop - 1) // 2
    STATE['busy'] = False
    redraw()


def on_group(*_):
    opts = GROUPS[w_group.value]
    keep = [n for n in w_lms.value if n in opts]
    aopts = ANGLE_GROUPS[w_group.value]
    akeep = [a for a in w_angles.value if a in aopts]
    mirrored = [a.replace('R ', 'L ', 1) for a in DEFAULT_ANGLES]
    STATE['busy'] = True
    w_lms.options = opts
    w_lms.value = keep or [n for n in DEFAULT_LANDMARKS if n in opts] or opts[:1]
    w_angles.options = aopts
    w_angles.value = (akeep or [a for a in DEFAULT_ANGLES if a in aopts] or [a for a in mirrored if a in aopts]
                      or aopts[:1])
    STATE['busy'] = False
    redraw()


def on_full(_):
    w_time.value = (w_time.min, w_time.max)


def report_errors(fn):
    """Show button errors in the notebook (otherwise ipywidgets hides them)."""
    def wrapper(*a):
        try:
            fn(*a)
        except Exception as e:
            w_msg.value = f'<span style="color:#d62728">error: {type(e).__name__}: {e}</span>'
    return wrapper


@report_errors
def save_html(_):
    if STATE['fig'] is None:
        return
    p = params()
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    f = EXPORT_DIR / (safe_name(f"{p['tr']['name']}_{tabs.titles[tabs.selected_index or 0]}_{p['quantity']}_"
                               f"{p['t_range'][0]:.2f}-{p['t_range'][1]:.2f}s") + '.html')
    STATE['fig'].write_html(f, include_plotlyjs='cdn')
    w_msg.value = f'plot saved: <code>{f}</code>'


@report_errors
def save_csv(_):
    p = params()
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    df, cols, involved = compute_series(**p)
    f = EXPORT_DIR / (safe_name(f"{p['tr']['name']}_{p['quantity']}_{p['t_range'][0]:.2f}-{p['t_range'][1]:.2f}s") + '.csv')
    df.to_csv(f, index=False)
    w_msg.value = f'data saved ({len(df)} rows, {len(cols)} series + class and failed checks per landmark): <code>{f}</code>'


w_rec.observe(on_recording, 'value')
w_group.observe(on_group, 'value')
w_time.observe(on_time, 'value')
for w in (w_lms, w_angles, w_qty, w_A, w_B, w_C, w_smooth, w_quality, w_suspect, w_raw, w_frame, w_heat, w_heat_group,
          w_mincams, w_reproj, w_dev, w_neargap, *w_axes):
    w.observe(redraw, 'value')
tabs.observe(redraw, 'selected_index')
w_full.on_click(on_full)
w_save_html.on_click(save_html)
w_save_csv.on_click(save_csv)

controls = W.HBox([
    W.VBox([w_group, w_lms, w_angles]),
    W.VBox([w_qty, w_axes_box, w_A, w_B, w_C, w_smooth, w_quality, w_suspect, w_raw]),
    w_thr,
])
ui = W.VBox([w_rec, controls, W.HBox([w_time, w_full]), w_info, tabs, W.HBox([w_save_html, w_save_csv]), w_msg])
display(ui)
on_recording()

## Use from code (without widgets)

The same functions can be called directly. This is handy for reproducible figures, for a report or for comparing
several recordings.

```python
tr = load_trial('louis_finertapping')               # part of the name is enough
classify(tr, min_cams=3, max_reproj_px=15)           # other thresholds (default: Parameters cell)
timeseries_figure(tr, ['R_index_tip'], 'speed (magnitude)', t_range=(2, 8), smooth=3).show()
df, cols, _ = compute_series(tr, [], 'distance A–B', A='R_thumb_tip', B='R_index_tip')   # DataFrame
timeseries_figure(tr, [], 'joint angles', angles=['R index MCP flexion', 'R index PIP flexion']).show()
angle_stats(tr, ANGLE_GROUPS['right hand'], t_range=(2, 8))     # quality + range of motion per angle
reliable = tr['cls'] == 3                            # (frame, 75) mask of the reliable points
```

| function | what it does |
|---|---|
| `find_trials()` | list of the recordings with 3D data available |
| `load_trial(name)` | data: `d3` (frame, 75, 3) mm, `raw`, `cls`, `reasons`, `ncams`, `reproj_max`, `reproj_mean`, `dev`, `near_gap`, `fps`, `t` |
| `classify(tr, min_cams, max_reproj_px, max_dev_mm, exclude_near_gap)` | recompute classes and failed checks with other thresholds |
| `compute_series(tr, landmarks, quantity, ...)` | DataFrame with series, class and failed checks of every point |
| `timeseries_figure(...)`, `trajectory_figure(...)`, `quality_heatmap(...)` | plotly figures |
| `stats_table(tr, landmarks, t_range)` | quality and movement table |
| `trial_angles(tr, smooth)` | all joint angles of both hands: `({name: (frames,) °}, {name: landmark indices})` |
| `angle_stats(tr, angles, t_range, smooth)` | quality and range of motion per joint angle |
| `ANGLE_NAMES`, `ANGLE_GROUPS` | available joint angles (e.g. `'R index PIP flexion'`) |

In [4]:
# example: reliability of all recordings (hands), with the thresholds of the Parameters cell
rows = []
for name in find_trials():
    tr = classify(load_trial(name))
    for hand, o in (('right', 33), ('left', 54)):
        c = tr['cls'][:, o:o + 21]
        rows.append({'recording': name, 'hand': hand, **{f'% {CLASS[k]}': 100 * (c == k).mean() for k in (3, 2, 1, 0)},
                     **{f'% {r}': 100 * tr['reasons'][r][:, o:o + 21].mean() for r in REASONS}})
pd.DataFrame(rows).set_index(['recording', 'hand']).round(1)

% reliable  % suspect  \
recording                                 hand                           
2026-11-10_111114_roberto_ninepeg_01      right         0.0       56.0   
                                          left          0.0       59.6   
2026-11-10_111157_roberto_ninepeg_01      right        98.1        1.9   
                                          left         92.1        6.5   
2026-11-10_111503_roberto_coin_01         right        94.3        5.2   
                                          left         98.7        1.2   
2026-11-10_111608_louis_ninepeg_01        right        89.1       10.9   
                                          left         95.9        4.1   
2026-11-10_111753_louis_coin_01           right        93.5        6.5   
                                          left         96.4        3.6   
2026-11-10_112048_louis_finertapping_01   right        96.2        3.8   
                                          left         97.3        2.7   
2026-11-10_112241_roberto_finertapping_01 right        98.3        1.7   
                                          left         98.9        1.1   

                                                 % interpolated  % missing  \
recording                                 hand                               
2026-11-10_111114_roberto_ninepeg_01      right             0.1       43.9   
                                          left              0.0       40.4   
2026-11-10_111157_roberto_ninepeg_01      right             0.0        0.0   
                                          left              0.0        1.4   
2026-11-10_111503_roberto_coin_01         right             0.0        0.5   
                                          left              0.1        0.0   
2026-11-10_111608_louis_ninepeg_01        right             0.0        0.0   
                                          left              0.0        0.0   
2026-11-10_111753_louis_coin_01           right             0.0        0.0   
                                          left              0.0        0.0   
2026-11-10_112048_louis_finertapping_01   right             0.0        0.0   
                                          left              0.0        0.0   
2026-11-10_112241_roberto_finertapping_01 right             0.0        0.0   
                                          left              0.0        0.0   

                                                 % few_cams  % reproj  \
recording                                 hand                          
2026-11-10_111114_roberto_ninepeg_01      right        56.0       0.1   
                                          left         59.5       0.1   
2026-11-10_111157_roberto_ninepeg_01      right         0.0       0.7   
                                          left          2.1       4.0   
2026-11-10_111503_roberto_coin_01         right         0.5       4.7   
                                          left          0.3       0.9   
2026-11-10_111608_louis_ninepeg_01        right         0.0       9.4   
                                          left          0.0       3.0   
2026-11-10_111753_louis_coin_01           right         0.0       6.3   
                                          left          0.0       3.4   
2026-11-10_112048_louis_finertapping_01   right         0.0       1.6   
                                          left          0.0       2.5   
2026-11-10_112241_roberto_finertapping_01 right         0.0       1.7   
                                          left          0.0       0.8   

                                                 % outlier  % near_gap  
recording                                 hand                          
2026-11-10_111114_roberto_ninepeg_01      right        5.1         1.1  
                                          left         0.6         0.4  
2026-11-10_111157_roberto_ninepeg_01      right        1.9         0.0  
                                          left         1.2         0.5  
2026-11-

In [5]:
# example: MCP and PIP flexion of all right-hand fingers, with quality, and their range of motion
tr = load_trial(rec0)
fingers = ['index', 'middle', 'ring', 'pinky']
timeseries_figure(tr, [], 'joint angles', angles=[f'R {f} {j} flexion' for f in fingers for j in ('MCP', 'PIP')],
                  height=450).show()
display(angle_stats(tr, ANGLE_GROUPS['right hand']))

,% reliable,% suspect,% interpolated,% missing,mean [°],SD [°],min [°],max [°],ROM [°],ROM p5–p95 [°],"mean, reliable [°]","ROM p5–p95, reliable [°]"
angle,,,,,,,,,,,,
R index MCP flexion,96.6,3.4,0.0,0.0,19.5,8.9,-0.2,48.3,48.5,35.0,19.3,34.5
R index PIP flexion,96.5,3.5,0.0,0.0,17.3,16.9,0.7,68.3,67.6,49.2,17.5,49.1
R index DIP flexion,96.4,3.6,0.0,0.0,5.0,9.1,-4.8,30.1,34.9,24.9,5.1,25.1
R index TAM,96.4,3.6,0.0,0.0,41.8,27.8,1.1,111.5,110.4,87.8,41.9,87.8
R middle MCP flexion,96.6,3.4,0.0,0.0,16.2,10.3,-10.1,49.5,59.6,37.4,16.0,37.0
R middle PIP flexion,95.8,4.2,0.0,0.0,19.5,20.4,2.2,69.8,67.7,59.1,19.6,59.4
R middle DIP flexion,94.0,6.0,0.0,0.0,6.0,11.5,-51.1,33.0,84.0,29.7,6.2,29.6
R middle TAM,94.0,6.0,0.0,0.0,41.6,33.3,-12.4,124.9,137.3,96.1,42.1,93.3
R ring MCP flexion,96.6,3.4,0.0,0.0,16.6,6.6,5.8,51.3,45.5,16.0,16.4,15.6


In [6]:
# example: right thumb–index distance in the first 10 s of a recording, with quality
tr = load_trial(rec0)
timeseries_figure(tr, [], 'distance A–B', A='R_thumb_tip', B='R_index_tip', t_range=(0, 10), height=350).show()
stats_table(tr, ['R_wrist', 'R_thumb_tip', 'R_index_tip', 'L_wrist', 'L_thumb_tip', 'L_index_tip'])

,% reliable,% suspect,% interpolated,% missing,% few_cams,% reproj,% outlier,% near_gap,mean cameras,median reproj max [px],p99 raw–final dev [mm],n gaps,max gap [s],range x [mm],range y [mm],range z [mm],mean speed [mm/s],max speed [mm/s]
landmark,,,,,,,,,,,,,,,,,,
R_wrist,97.29,2.71,0.0,0.0,0.0,0.99,2.53,0.0,7.75,4.86,33.58,0,0.0,109.78,43.56,55.89,39.03,1289.30
R_thumb_tip,96.02,3.98,0.0,0.0,0.0,2.17,2.62,0.0,7.75,3.45,24.56,0,0.0,127.76,27.81,119.71,71.62,986.64
R_index_tip,96.65,3.35,0.0,0.0,0.0,0.90,3.16,0.0,7.75,2.53,49.65,0,0.0,180.75,64.57,113.76,90.85,2055.12
L_wrist,98.73,1.27,0.0,0.0,0.0,0.72,1.08,0.0,7.86,5.28,10.40,0,0.0,92.62,49.74,76.35,25.97,500.98
L_thumb_tip,99.37,0.63,0.0,0.0,0.0,0.63,0.45,0.0,7.86,5.64,6.02,0,0.0,96.80,49.46,143.90,81.10,837.34
L_index_tip,89.24,10.76,0.0,0.0,0.0,10.49,0.45,0.0,7.86,3.55,6.48,0,0.0,93.12,78.30,138.06,84.40,642.59
